# ПР-8. Градиентный бустинг и сравнение моделей (уровень B)
**Задача:** предсказать отток абонентов (`churn`). Сравниваем логистическую регрессию, случайный лес и градиентный бустинг на **одном и том же разбиении**.

Библиотека бустинга: `HistGradientBoostingClassifier` из scikit-learn (подсказка из задания — принцип тот же, что у XGBoost/LightGBM, есть ранняя остановка). Если у вас установлен XGBoost, можно заменить модель — логика останется прежней.

In [ ]:
import time
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

print("scikit-learn", sklearn.__version__)
RS = 42   # один random_state для всего

## 1. Данные и разбиение

In [ ]:
df = pd.read_csv("o_churn_synthetic.csv")      # путь поправьте под себя
X = df.drop(columns=["churn", "subscriber_code"])   # id абонента — не признак
y = df["churn"]

cat_cols = ["region", "tariff"]
num_cols = [c for c in X.columns if c not in cat_cols]

# Одно разбиение для ВСЕХ моделей (если в ПР-7 было другое — возьмите его параметры)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y)

# Из train выделяем валидацию для ранней остановки. Тест не трогаем.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RS, stratify=y_train)

print(X_train.shape, X_test.shape, "| доля оттока:", round(y.mean(), 3))

## 2. Подготовка признаков
Категории (`region`, `tariff`) кодируем one-hot, числа оставляем как есть (для логрегрессии ещё масштабируем).

In [ ]:
def prep(scale):
    nums = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", nums, num_cols)])

# Для бустинга готовим матрицы один раз
prep_gb = prep(scale=False).fit(X_fit)
Xf, Xv = prep_gb.transform(X_fit), prep_gb.transform(X_val)

## 3. Бустинг с ранней остановкой
Ставим много деревьев (`max_iter=1000`), но обучение само остановится, когда качество на **валидации** перестанет расти.
В scikit-learn валидация для ранней остановки выделяется автоматически из обучающей выборки (`validation_fraction=0.2`), тест при этом не используется.

In [ ]:
gb = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=3, max_iter=1000,
    early_stopping=True, validation_fraction=0.2,
    n_iter_no_change=20, random_state=RS)
gb.fit(prep_gb.transform(X_train), y_train)
print("Оптимальное число деревьев:", gb.n_iter_)

## 4. Подбор learning_rate и max_depth
Перебираем несколько значений. Выбираем по **валидации**, а не по тесту.

In [ ]:
rows = []
for lr in [0.3, 0.1, 0.03, 0.01]:
    for depth in [2, 3, 5]:
        t = time.time()
        m = HistGradientBoostingClassifier(
            learning_rate=lr, max_depth=depth, max_iter=2000,
            early_stopping=True, validation_fraction=0.2,
            n_iter_no_change=20, random_state=RS)
        m.fit(Xf, y_fit)
        rows.append({"learning_rate": lr, "max_depth": depth,
                     "деревьев": m.n_iter_,
                     "AUC валидации": round(roc_auc_score(y_val, m.predict_proba(Xv)[:, 1]), 4),
                     "время, с": round(time.time() - t, 2)})
grid = pd.DataFrame(rows).sort_values("AUC валидации", ascending=False)
grid

In [ ]:
best = grid.iloc[0]
print("Лучшие параметры:", best.to_dict())
print()
print("Средний AUC валидации по learning_rate:")
print(grid.groupby("learning_rate")[["AUC валидации", "деревьев", "время, с"]].mean().round(4))

**Как читать:** чем меньше `learning_rate`, тем больше деревьев нужно и тем дольше обучение. Качество при этом растёт лишь до некоторого предела (потом выходит на плато) — это компромисс «качество ↔ время».

## 5. Итоговое сравнение трёх моделей

In [ ]:
best_lr, best_depth = float(best["learning_rate"]), int(best["max_depth"])

models = {
    "Логистическая регрессия": make_pipeline(prep(True), LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(prep(False), RandomForestClassifier(
        n_estimators=300, random_state=RS, n_jobs=-1)),
    "Бустинг (с ранней остановкой)": make_pipeline(prep(False), HistGradientBoostingClassifier(
        learning_rate=best_lr, max_depth=best_depth, max_iter=2000,
        early_stopping=True, validation_fraction=0.2,
        n_iter_no_change=20, random_state=RS)),
}

res = []
for name, model in models.items():
    t = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t
    res.append({"Модель": name,
                "AUC train": round(roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]), 4),
                "AUC test":  round(roc_auc_score(y_test,  model.predict_proba(X_test)[:, 1]), 4),
                "Accuracy test": round(accuracy_score(y_test, model.predict(X_test)), 4),
                "Время обучения, с": round(fit_time, 2)})
result = pd.DataFrame(res)
result

## 6. Вывод
(Числа могут слегка отличаться на вашем компьютере — сверьтесь с таблицей.)

- **Качество:** у меня лучший AUC на тесте у логистической регрессии (≈0.81), затем бустинг (≈0.79) и случайный лес (≈0.76). Разница между логрегрессией и бустингом небольшая — тест всего 600 строк, так что это почти ничья.
- **Переобучение:** у случайного леса AUC на train = 1.00, а на тесте 0.76 — он запомнил обучающие данные. У бустинга с ранней остановкой разрыв намного меньше.
- **Время:** логрегрессия быстрее всех, лес медленнее всех.
- **Выбор:** на этих данных зависимости, похоже, близки к линейным, поэтому простая логистическая регрессия не уступает бустингу и при этом быстрее и понятнее. Бустинг стоит брать, если нужно уловить сложные нелинейные связи.

**Отличие бустинга от случайного леса (абзац для работы):** случайный лес строит много деревьев *независимо* друг от друга на случайных подвыборках и усредняет их ответы — это снижает разброс. Бустинг строит деревья *последовательно*: каждое следующее учится исправлять ошибки уже построенного ансамбля (по градиенту функции потерь), поэтому деревья неглубокие и «слабые», а качество растёт за счёт их накопления; за это отвечают `learning_rate` и число деревьев, а переобучение ограничивают ранней остановкой.